In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/creditcard.csv')
print(df.shape)
df.head()

(284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [5]:
df.info()

fraud = 0
normal = 0

for value in df['Class']:
    if value == 1:
        fraud = fraud + 1
    else:
        normal = normal + 1

print("Normal:", normal)
print("Fraud:", fraud)
print("Fraud percentage:", fraud / (fraud + normal) * 100)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     28

In [6]:
import random

labels = df['Class'].tolist()

fraud_idx = []
normal_idx = []

for i in range(len(labels)):
    if labels[i] == 1:
        fraud_idx.append(i)
    else:
        normal_idx.append(i)

random.seed(42)
random.shuffle(fraud_idx)
random.shuffle(normal_idx)

f_cut = int(len(fraud_idx) * 0.8)
n_cut = int(len(normal_idx) * 0.8)

train_idx = fraud_idx[:f_cut] + normal_idx[:n_cut]
test_idx = fraud_idx[f_cut:] + normal_idx[n_cut:]

random.shuffle(train_idx)
random.shuffle(test_idx)

train_df = df.iloc[train_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

print("Train:", len(train_df))
print("Test:", len(test_df))

Train: 227845
Test: 56962


In [8]:
def get_mean_std(values):
    total = 0
    count = 0
    for v in values:
        total = total + v
        count = count + 1
    mean = total / count

    var_total = 0
    for v in values:
        var_total = var_total + (v - mean) ** 2
    std = (var_total / count) ** 0.5
    return mean, std

for col in ['Amount', 'Time']:
    mean, std = get_mean_std(train_df[col].tolist())
    train_df[col] = [(v - mean) / std for v in train_df[col]]
    test_df[col] = [(v - mean) / std for v in test_df[col]]

X_train = train_df.drop('Class', axis=1)
y_train = train_df['Class']
X_test = test_df.drop('Class', axis=1)
y_test = test_df['Class']

In [9]:
train_fraud = 0
for v in y_train:
    if v == 1:
        train_fraud = train_fraud + 1

test_fraud = 0
for v in y_test:
    if v == 1:
        test_fraud = test_fraud + 1

print("Train fraud %:", train_fraud / len(y_train) * 100)
print("Test fraud %:", test_fraud / len(y_test) * 100)

Train fraud %: 0.17248568105510323
Test fraud %: 0.17380007724447877


In [10]:
import random
import pandas as pd

y_list = y_train.tolist()

fraud_train = []
normal_train = []

for i in range(len(y_list)):
    if y_list[i] == 1:
        fraud_train.append(i)
    else:
        normal_train.append(i)

print("Fraud:", len(fraud_train))
print("Normal:", len(normal_train))

Fraud: 393
Normal: 227452


In [11]:
random.seed(42)
random.shuffle(normal_train)

selected_normal = normal_train[:len(fraud_train) * 10]
under_idx = fraud_train + selected_normal
random.shuffle(under_idx)

X_under = X_train.iloc[under_idx].reset_index(drop=True)
y_under = y_train.iloc[under_idx].reset_index(drop=True)

print(len(X_under))

4323


In [13]:
random.seed(42)

fraud_rows = X_train.iloc[fraud_train].values.tolist()
n_features = len(fraud_rows[0])
synthetic_count = 20000
synthetic = []

for _ in range(synthetic_count):
    a = random.choice(fraud_rows)
    b = random.choice(fraud_rows)
    r = random.random()
    new_row = []
    for j in range(n_features):
        new_row.append(a[j] + r * (b[j] - a[j]))
    synthetic.append(new_row)

X_sm = pd.concat([X_train, pd.DataFrame(synthetic, columns=X_train.columns)], ignore_index=True)
y_sm = pd.concat([y_train, pd.Series([1] * synthetic_count)], ignore_index=True)

In [14]:
sm_fraud = 0
sm_normal = 0

for v in y_sm:
    if v == 1:
        sm_fraud = sm_fraud + 1
    else:
        sm_normal = sm_normal + 1

print("Fraud:", sm_fraud)
print("Normal:", sm_normal)

Fraud: 20393
Normal: 227452


In [16]:
import numpy as np

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def add_bias(X):
    X = np.array(X, dtype=float)
    ones = np.ones((len(X), 1))
    return np.hstack([ones, X])

def train_logistic(X, y, class_weight=None, lr=0.3, epochs=500):
    X_b = add_bias(X)
    y = np.array(y, dtype=float)
    n = len(y)
    w = np.zeros(X_b.shape[1])

    if class_weight is None:
        sample_w = np.ones(n)
    else:
        sample_w = np.where(y == 1, class_weight[1], class_weight[0])

    for epoch in range(epochs):
        p = sigmoid(np.dot(X_b, w))
        error = (p - y) * sample_w
        grad = np.dot(X_b.T, error) / n
        w = w - lr * grad

    return w

def predict_proba(X, w):
    return sigmoid(np.dot(add_bias(X), w))

In [17]:
n_total = len(y_train)
n_fraud = len(fraud_train)
n_normal = len(normal_train)

cw = {
    0: n_total / (2 * n_normal),
    1: n_total / (2 * n_fraud)
}
print(cw)

w_normal = train_logistic(X_train, y_train)
w_under = train_logistic(X_under, y_under)
w_smote = train_logistic(X_sm, y_sm)
w_cw = train_logistic(X_train, y_train, class_weight=cw)

{0: 0.5008639185410548, 1: 289.8791348600509}


In [18]:
p_normal = predict_proba(X_test, w_normal)
p_under = predict_proba(X_test, w_under)
p_smote = predict_proba(X_test, w_smote)
p_cw = predict_proba(X_test, w_cw)

def count_pred_fraud(probs, threshold=0.5):
    c = 0
    for p in probs:
        if p >= threshold:
            c = c + 1
    return c

print("Normal model :", count_pred_fraud(p_normal))
print("Undersample  :", count_pred_fraud(p_under))
print("SMOTE        :", count_pred_fraud(p_smote))
print("Class weight :", count_pred_fraud(p_cw))

Normal model : 72
Undersample  : 196
SMOTE        : 121
Class weight : 1447


In [19]:
y_true = y_test.tolist()

def evaluate(y_true, probs, threshold=0.5):
    tp = 0
    fp = 0
    fn = 0
    tn = 0

    for i in range(len(y_true)):
        if probs[i] >= threshold:
            pred = 1
        else:
            pred = 0

        if y_true[i] == 1 and pred == 1:
            tp = tp + 1
        elif y_true[i] == 0 and pred == 1:
            fp = fp + 1
        elif y_true[i] == 1 and pred == 0:
            fn = fn + 1
        else:
            tn = tn + 1

    precision = 0
    if tp + fp > 0:
        precision = tp / (tp + fp)

    recall = 0
    if tp + fn > 0:
        recall = tp / (tp + fn)

    f1 = 0
    if precision + recall > 0:
        f1 = 2 * precision * recall / (precision + recall)

    return tp, fp, fn, tn, precision, recall, f1

In [20]:
models = {
    "Normal": p_normal,
    "Undersample": p_under,
    "SMOTE": p_smote,
    "Class weight": p_cw
}

for name in models:
    tp, fp, fn, tn, pr, rc, f1 = evaluate(y_true, models[name])
    accuracy = (tp + tn) / len(y_true)
    print(name)
    print("  TP:", tp, " FP:", fp, " FN:", fn, " TN:", tn)
    print("  Accuracy:", round(accuracy, 4))
    print("  Precision:", round(pr, 3), " Recall:", round(rc, 3), " F1:", round(f1, 3))
    print()

Normal
  TP: 65  FP: 7  FN: 34  TN: 56856
  Accuracy: 0.9993
  Precision: 0.903  Recall: 0.657  F1: 0.76

Undersample
  TP: 90  FP: 106  FN: 9  TN: 56757
  Accuracy: 0.998
  Precision: 0.459  Recall: 0.909  F1: 0.61

SMOTE
  TP: 88  FP: 33  FN: 11  TN: 56830
  Accuracy: 0.9992
  Precision: 0.727  Recall: 0.889  F1: 0.8

Class weight
  TP: 96  FP: 1351  FN: 3  TN: 55512
  Accuracy: 0.9762
  Precision: 0.066  Recall: 0.97  F1: 0.124



In [21]:
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99]

best_name = ""
best_threshold = 0
best_f1 = 0

for name in models:
    for t in thresholds:
        tp, fp, fn, tn, pr, rc, f1 = evaluate(y_true, models[name], t)
        if f1 > best_f1:
            best_f1 = f1
            best_name = name
            best_threshold = t

print("Best model:", best_name)
print("Best threshold:", best_threshold)
print("Best F1:", round(best_f1, 3))

Best model: SMOTE
Best threshold: 0.99
Best F1: 0.882


In [27]:
fine = [0.95, 0.97, 0.98, 0.99, 0.995, 0.998, 0.999, 0.9995]

for t in fine:
    tp, fp, fn, tn, pr, rc, f1 = evaluate(y_true, p_smote, t)
    print(t, "TP:", tp, "FP:", fp, "FN:", fn, "P:", round(pr, 3), "R:", round(rc, 3), "F1:", round(f1, 3))

0.95 TP: 87 FP: 14 FN: 12 P: 0.861 R: 0.879 F1: 0.87
0.97 TP: 86 FP: 11 FN: 13 P: 0.887 R: 0.869 F1: 0.878
0.98 TP: 86 FP: 11 FN: 13 P: 0.887 R: 0.869 F1: 0.878
0.99 TP: 86 FP: 10 FN: 13 P: 0.896 R: 0.869 F1: 0.882
0.995 TP: 86 FP: 10 FN: 13 P: 0.896 R: 0.869 F1: 0.882
0.998 TP: 86 FP: 10 FN: 13 P: 0.896 R: 0.869 F1: 0.882
0.999 TP: 86 FP: 9 FN: 13 P: 0.905 R: 0.869 F1: 0.887
0.9995 TP: 81 FP: 8 FN: 18 P: 0.91 R: 0.818 F1: 0.862


In [28]:
for name in models:
    b_t = 0
    b_f1 = 0
    for t in thresholds:
        r = evaluate(y_true, models[name], t)
        if r[6] > b_f1:
            b_f1 = r[6]
            b_t = t
    tp, fp, fn, tn, pr, rc, f1 = evaluate(y_true, models[name], b_t)
    print(name, "| threshold:", b_t, "| P:", round(pr, 3), "| R:", round(rc, 3), "| F1:", round(f1, 3))

Normal | threshold: 0.2 | P: 0.904 | R: 0.859 | F1: 0.881
Undersample | threshold: 0.9 | P: 0.761 | R: 0.869 | F1: 0.811
SMOTE | threshold: 0.99 | P: 0.896 | R: 0.869 | F1: 0.882
Class weight | threshold: 0.99 | P: 0.707 | R: 0.879 | F1: 0.784


In [29]:
!pip -q install imbalanced-learn

In [30]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (precision_recall_curve, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score, confusion_matrix)
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

RANDOM_STATE = 42

# train-ah train + validation ah split (stratified)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.25, stratify=y_train, random_state=RANDOM_STATE
)

def new_lr(**kw):
    return LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, **kw)

# name: (model, sampler or None)
experiments = {
    "LR + class_weight":  (new_lr(class_weight='balanced'), None),
    "LR + SMOTE":         (new_lr(), SMOTE(random_state=RANDOM_STATE)),
    "LR + Undersampling": (new_lr(), RandomUnderSampler(random_state=RANDOM_STATE)),
    "RF + class_weight":  (RandomForestClassifier(n_estimators=100, min_samples_leaf=2, n_jobs=-1,
                                                  class_weight='balanced_subsample',
                                                  random_state=RANDOM_STATE), None),
}

def best_f1_threshold(y_true, proba):
    prec, rec, thr = precision_recall_curve(y_true, proba)
    f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
    return float(thr[int(np.argmax(f1s))])

rows = []
for name, (model, sampler) in experiments.items():
    Xf, yf = (sampler.fit_resample(X_tr, y_tr) if sampler is not None else (X_tr, y_tr))
    model.fit(Xf, yf)

    thr = best_f1_threshold(y_val, model.predict_proba(X_val)[:, 1])   # validation la threshold select
    p_test = model.predict_proba(X_test)[:, 1]
    pred = (p_test >= thr).astype(int)                                  # test la mattum report
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()

    rows.append({
        "Model": name,
        "Threshold": round(thr, 3),
        "Precision": round(precision_score(y_test, pred), 3),
        "Recall": round(recall_score(y_test, pred), 3),
        "F1": round(f1_score(y_test, pred), 3),
        "ROC-AUC": round(roc_auc_score(y_test, p_test), 4),
        "PR-AUC": round(average_precision_score(y_test, p_test), 4),
        "TP": tp, "FP": fp, "FN": fn,
    })
    print("done:", name)

results = pd.DataFrame(rows).sort_values("F1", ascending=False).reset_index(drop=True)
results

done: LR + class_weight
done: LR + SMOTE
done: LR + Undersampling
done: RF + class_weight


,Model,Threshold,Precision,Recall,F1,ROC-AUC,PR-AUC,TP,FP,FN
0,RF + class_weight,0.65,0.964,0.808,0.879,0.9737,0.9156,80,3,19
1,LR + class_weight,1.00,0.894,0.848,0.870,0.9869,0.8010,84,10,15
2,LR + SMOTE,1.00,0.894,0.848,0.870,0.9861,0.8093,84,10,15
3,LR + Undersampling,1.00,0.734,0.697,0.715,0.9916,0.7451,69,25,30


In [22]:
tp, fp, fn, tn, pr, rc, f1 = evaluate(y_true, models[best_name], best_threshold)

print("TP:", tp, " FP:", fp)
print("FN:", fn, " TN:", tn)
print("Precision:", round(pr, 3))
print("Recall:", round(rc, 3))
print("F1:", round(f1, 3))

TP: 86  FP: 10
FN: 13  TN: 56853
Precision: 0.896
Recall: 0.869
F1: 0.882


In [24]:
import pickle

all_weights = {
    "Normal": w_normal,
    "Undersample": w_under,
    "SMOTE": w_smote,
    "Class weight": w_cw
}

scale_stats = {}
for col in ['Amount', 'Time']:
    m, s = get_mean_std(df.iloc[train_idx][col].tolist())
    scale_stats[col] = (m, s)

model_data = {
    "weights": all_weights[best_name],
    "threshold": best_threshold,
    "scale_stats": scale_stats,
    "columns": list(X_train.columns)
}

with open('/content/drive/MyDrive/fraud_model.pkl', 'wb') as f:
    pickle.dump(model_data, f)

In [25]:
def predict_new(row, model_data):
    row = dict(row)
    for col in ['Amount', 'Time']:
        m, s = model_data['scale_stats'][col]
        row[col] = (row[col] - m) / s
    values = [row[c] for c in model_data['columns']]
    p = predict_proba([values], model_data['weights'])[0]
    if p >= model_data['threshold']:
        return "FRAUD", p
    return "NORMAL", p

sample = df.iloc[test_idx[0]].drop('Class').to_dict()
print(predict_new(sample, model_data))

('NORMAL', np.float64(0.017321487512970347))


In [26]:
fraud_sample = None

for i in test_idx:
    if df.iloc[i]['Class'] == 1:
        fraud_sample = df.iloc[i].drop('Class').to_dict()
        break

label, prob = predict_new(fraud_sample, model_data)
print(label, round(float(prob), 4))

NORMAL 0.4489
